# MathScy: State-of-the-Art Mixture-of-Experts (MoE) Approaches — Reference Implementation

This notebook implements **practical, grant-ready MoE patterns** tailored to the **MathScy** architecture described in the attached NSF proposal (AIMing: Interactive Conjecture Proving): modular **specialist experts** for conjecture/proof/counterexample workflows, with **tool use** and **human-facing explanations**.

What you'll get:
1. **Sparse token-level MoE (Switch / Top-1 routing)** with load-balancing loss.
2. **Expert-Choice routing** (experts pull tokens) as a modern alternative to Top-k gating.
3. **Parameter-efficient MoE via LoRA-style experts** (Mixture-of-Adapters) for feasible fine-tuning.
4. **Tool-augmented MoE router** for Algebraic Geometry workflows (SymPy tools as “experts”), matching MathScy’s “LLM + external tools” design.

> The implementations are intentionally **small and CPU-friendly** so they can run in this environment, while preserving the same control surfaces (routing, capacity, balance loss, adapter experts) used at scale.

## 0. Setup

In [ ]:
import math
import random
import re
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Optional (used later for tool-expert demos)
import sympy as sp

def set_seed(seed: int = 7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(7)
device = torch.device("cpu")
print("torch:", torch.__version__, "| sympy:", sp.__version__, "| device:", device)

## 1) Sparse MoE (Switch / Top-1 routing)

**Why this matters for MathScy**: you can allocate compute to *specialist experts* (e.g., conjecture vs proof vs counterexample) without paying the cost of a fully dense model.  
This aligns with the proposal’s goal of *scalability* via dynamic routing.

We implement:
- **Top-1 routing** per token
- **capacity factor** (to avoid overload)
- **load-balancing auxiliary loss** (encourages even expert utilization)

In [ ]:
@dataclass
class MoEStats:
    aux_loss: torch.Tensor
    expert_load: torch.Tensor         # number of tokens routed to each expert
    expert_importance: torch.Tensor   # normalized prob mass per expert

class FeedForwardExpert(nn.Module):
    def __init__(self, d_model: int, d_hidden: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_hidden)
        self.w2 = nn.Linear(d_hidden, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.gelu(self.w1(x)))

class Top1Router(nn.Module):
    '''
    Switch-style Top-1 gating:
      - Each token selects a single expert.
      - capacity_factor controls max tokens per expert.
    '''
    def __init__(self, d_model: int, n_experts: int, capacity_factor: float = 1.25):
        super().__init__()
        self.n_experts = n_experts
        self.capacity_factor = capacity_factor
        self.gate = nn.Linear(d_model, n_experts)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, MoEStats]:
        logits = self.gate(x)                         # [N,E]
        probs = F.softmax(logits, dim=-1)             # [N,E]

        top1_idx = torch.argmax(probs, dim=-1)        # [N]
        dispatch = F.one_hot(top1_idx, num_classes=self.n_experts).float()

        N = x.shape[0]
        cap = int(self.capacity_factor * math.ceil(N / self.n_experts))

        kept = torch.zeros_like(dispatch)
        expert_load = torch.zeros(self.n_experts, device=x.device)

        # Capacity: keep earliest cap tokens per expert (simple policy)
        for e in range(self.n_experts):
            token_ids = torch.nonzero(dispatch[:, e] > 0.0, as_tuple=False).squeeze(-1)
            if token_ids.numel() == 0:
                continue
            token_ids = token_ids[:cap]
            kept[token_ids, e] = 1.0
            expert_load[e] = token_ids.numel()

        combine = probs * kept

        # Switch auxiliary loss: encourage balanced importance/load
        importance = probs.sum(dim=0)  # [E]
        load = kept.sum(dim=0)         # [E]
        importance = importance / (importance.sum() + 1e-9)
        load = load / (load.sum() + 1e-9)
        aux_loss = (importance * load).sum() * (self.n_experts ** 2)

        return kept, combine, MoEStats(aux_loss=aux_loss, expert_load=expert_load, expert_importance=importance.detach())

class SwitchMoELayer(nn.Module):
    def __init__(self, d_model: int, d_hidden: int, n_experts: int, capacity_factor: float = 1.25):
        super().__init__()
        self.router = Top1Router(d_model, n_experts, capacity_factor)
        self.experts = nn.ModuleList([FeedForwardExpert(d_model, d_hidden) for _ in range(n_experts)])

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, MoEStats]:
        B, T, D = x.shape
        tokens = x.reshape(B*T, D)  # [N,D]
        dispatch, combine, stats = self.router(tokens)

        y_tokens = torch.zeros_like(tokens)
        for e, expert in enumerate(self.experts):
            idx = torch.nonzero(dispatch[:, e] > 0.0, as_tuple=False).squeeze(-1)
            if idx.numel() == 0:
                continue
            out_e = expert(tokens[idx])
            w = combine[idx, e].unsqueeze(-1)
            y_tokens[idx] += out_e * w

        return y_tokens.reshape(B, T, D), stats

### 1.1 Demo: Multi-function regression (experts specialize by task)

We create a toy dataset where each sample belongs to one of several functions.  
A small model with a **SwitchMoE** layer learns:
- to route similar samples to the same expert
- to specialize each expert

This mirrors MathScy routing different user requests to conjecture/proof/counterexample specialists.

In [ ]:
def sample_batch(batch_size: int, n_tasks: int = 4):
    '''
    Returns:
      x:    [B, 2] inputs
      task: [B]    task id
      y:    [B, 1] target
    '''
    x = torch.rand(batch_size, 2) * 2 - 1  # [-1,1]
    task = torch.randint(0, n_tasks, (batch_size,))
    y = torch.zeros(batch_size, 1)

    for i in range(batch_size):
        a, b = x[i]
        t = int(task[i])
        if t == 0:
            y[i] = torch.sin(3*a) + 0.3*b
        elif t == 1:
            y[i] = a*b + 0.2*a
        elif t == 2:
            y[i] = a*a + b*b
        elif t == 3:
            y[i] = a - b + 0.1*torch.cos(5*b)
    return x, task, y

class TinyMoERegressor(nn.Module):
    def __init__(self, d_model=48, d_hidden=96, n_experts=4):
        super().__init__()
        self.n_experts = n_experts
        self.embed = nn.Linear(2 + n_experts, d_model)  # append one-hot task token
        self.moe = SwitchMoELayer(d_model, d_hidden, n_experts=n_experts, capacity_factor=1.25)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x2: torch.Tensor, task: torch.Tensor):
        task_oh = F.one_hot(task, num_classes=self.n_experts).float()
        inp = torch.cat([x2, task_oh], dim=-1).unsqueeze(1)  # [B,1,2+E]
        h = self.embed(inp)                                  # [B,1,D]
        h, stats = self.moe(h)                               # [B,1,D]
        yhat = self.head(h).squeeze(1)                       # [B,1]
        return yhat, stats

set_seed(7)
model = TinyMoERegressor().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)

def train_steps(steps=400, batch_size=256, aux_weight=0.01):
    model.train()
    for step in range(steps):
        x, task, y = sample_batch(batch_size, n_tasks=4)
        x, task, y = x.to(device), task.to(device), y.to(device)

        yhat, stats = model(x, task)
        mse = F.mse_loss(yhat, y)
        loss = mse + aux_weight * stats.aux_loss

        opt.zero_grad()
        loss.backward()
        opt.step()

        if (step+1) % 100 == 0:
            print(f"step {step+1:4d} | mse {mse.item():.4f} | aux {stats.aux_loss.item():.4f} | load {stats.expert_load.tolist()}")

train_steps()

In [ ]:
model.eval()
with torch.no_grad():
    x, task, y = sample_batch(2000, n_tasks=4)
    yhat, _ = model(x.to(device), task.to(device))
    mse = F.mse_loss(yhat.cpu(), y)
    print("eval mse:", float(mse))

# Inspect router usage
with torch.no_grad():
    x, task, _ = sample_batch(4096, n_tasks=4)
    task_oh = F.one_hot(task, num_classes=model.n_experts).float()
    inp = torch.cat([x, task_oh], dim=-1).unsqueeze(1)
    h = model.embed(inp).reshape(-1, model.embed.out_features)
    logits = model.moe.router.gate(h)
    top1 = torch.argmax(logits, dim=-1)
    counts = torch.bincount(top1, minlength=model.n_experts).cpu().numpy()
    print("router top-1 counts:", counts, "| fraction:", counts / counts.sum())

## 2) Expert-Choice Routing (experts pull tokens)

**Expert-Choice** routing flips the selection:
- each expert picks the tokens it wants (up to capacity)
- yields more stable utilization and can reduce routing pathologies

Below is a compact implementation.

In [ ]:
class ExpertChoiceRouter(nn.Module):
    '''
    Expert-Choice routing:
      - compute scores s[t,e] for token t and expert e
      - each expert selects top 'capacity' tokens
      - each token is assigned to (at most) one expert by picking its best selected expert
    '''
    def __init__(self, d_model: int, n_experts: int, capacity_factor: float = 1.25):
        super().__init__()
        self.n_experts = n_experts
        self.capacity_factor = capacity_factor
        self.gate = nn.Linear(d_model, n_experts)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, MoEStats]:
        logits = self.gate(x)                # [N,E]
        scores = F.softmax(logits, dim=-1)   # [N,E]

        N = x.shape[0]
        cap = int(self.capacity_factor * math.ceil(N / self.n_experts))

        selected = torch.zeros_like(scores)
        expert_load = torch.zeros(self.n_experts, device=x.device)

        for e in range(self.n_experts):
            vals = scores[:, e]
            topk = torch.topk(vals, k=min(cap, N), sorted=False).indices
            selected[topk, e] = 1.0
            expert_load[e] = topk.numel()

        masked_scores = scores * selected + (-1e9) * (1 - selected)
        best_e = torch.argmax(masked_scores, dim=-1)  # [N]
        dispatch = F.one_hot(best_e, num_classes=self.n_experts).float()

        # tokens not selected by any expert -> drop
        has_any = (selected.sum(dim=-1) > 0).float().unsqueeze(-1)
        dispatch = dispatch * has_any
        combine = scores * dispatch

        importance = scores.sum(dim=0)
        load = dispatch.sum(dim=0)
        importance = importance / (importance.sum() + 1e-9)
        load = load / (load.sum() + 1e-9)
        aux_loss = (importance * load).sum() * (self.n_experts ** 2)

        return dispatch, combine, MoEStats(aux_loss=aux_loss, expert_load=expert_load, expert_importance=importance.detach())

class ExpertChoiceMoELayer(nn.Module):
    def __init__(self, d_model: int, d_hidden: int, n_experts: int, capacity_factor: float = 1.25):
        super().__init__()
        self.router = ExpertChoiceRouter(d_model, n_experts, capacity_factor)
        self.experts = nn.ModuleList([FeedForwardExpert(d_model, d_hidden) for _ in range(n_experts)])

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, MoEStats]:
        B, T, D = x.shape
        tokens = x.reshape(B*T, D)
        dispatch, combine, stats = self.router(tokens)

        y_tokens = torch.zeros_like(tokens)
        for e, expert in enumerate(self.experts):
            idx = torch.nonzero(dispatch[:, e] > 0.0, as_tuple=False).squeeze(-1)
            if idx.numel() == 0:
                continue
            out_e = expert(tokens[idx])
            w = combine[idx, e].unsqueeze(-1)
            y_tokens[idx] += out_e * w

        return y_tokens.reshape(B, T, D), stats

### 2.1 Compare Switch vs Expert-Choice on the same toy task

In [ ]:
class TinyMoERegressorExpertChoice(nn.Module):
    def __init__(self, d_model=48, d_hidden=96, n_experts=4):
        super().__init__()
        self.n_experts = n_experts
        self.embed = nn.Linear(2 + n_experts, d_model)
        self.moe = ExpertChoiceMoELayer(d_model, d_hidden, n_experts=n_experts, capacity_factor=1.25)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x2: torch.Tensor, task: torch.Tensor):
        task_oh = F.one_hot(task, num_classes=self.n_experts).float()
        inp = torch.cat([x2, task_oh], dim=-1).unsqueeze(1)
        h = self.embed(inp)
        h, stats = self.moe(h)
        yhat = self.head(h).squeeze(1)
        return yhat, stats

set_seed(7)
model_ec = TinyMoERegressorExpertChoice().to(device)
opt_ec = torch.optim.AdamW(model_ec.parameters(), lr=2e-3)

def train_ec(steps=400, batch_size=256, aux_weight=0.01):
    model_ec.train()
    for step in range(steps):
        x, task, y = sample_batch(batch_size, n_tasks=4)
        yhat, stats = model_ec(x, task)
        mse = F.mse_loss(yhat, y)
        loss = mse + aux_weight * stats.aux_loss
        opt_ec.zero_grad()
        loss.backward()
        opt_ec.step()
        if (step+1) % 100 == 0:
            print(f"step {step+1:4d} | mse {mse.item():.4f} | aux {stats.aux_loss.item():.4f} | load {stats.expert_load.tolist()}")

train_ec()

model_ec.eval()
with torch.no_grad():
    x, task, y = sample_batch(2000, n_tasks=4)
    yhat, stats = model_ec(x, task)
    mse = F.mse_loss(yhat, y)
    print("eval mse:", float(mse), "| last-step load:", stats.expert_load.tolist())

## 3) Parameter-Efficient MoE: Mixture-of-LoRA-Experts (MoLE)

A practical approach is to keep a **shared backbone** and attach multiple **LoRA-style adapter experts**.
Only the selected adapter is applied per token — MoE behavior with **tiny per-expert parameter cost**.

Below: a minimal LoRA linear module + an MoE layer that routes tokens to different LoRA experts.

In [ ]:
class LoRALinear(nn.Module):
    '''
    Linear layer with LoRA adaptation:
      y = xW^T + (alpha/r) * x A B
    '''
    def __init__(self, in_features: int, out_features: int, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base = nn.Linear(in_features, out_features)
        self.r = r
        self.alpha = alpha
        self.A = nn.Parameter(torch.zeros(in_features, r))
        self.B = nn.Parameter(torch.zeros(r, out_features))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = self.base(x)
        lora = (x @ self.A) @ self.B
        return base + (self.alpha / self.r) * lora

class LoRAExpertFFN(nn.Module):
    def __init__(self, d_model: int, d_hidden: int, r: int = 8, alpha: float = 16.0):
        super().__init__()
        self.w1 = LoRALinear(d_model, d_hidden, r=r, alpha=alpha)
        self.w2 = LoRALinear(d_hidden, d_model, r=r, alpha=alpha)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.gelu(self.w1(x)))

class LoRA_MoE_Layer(nn.Module):
    def __init__(self, d_model: int, d_hidden: int, n_experts: int, r: int = 8, alpha: float = 16.0, capacity_factor: float = 1.25):
        super().__init__()
        self.router = Top1Router(d_model, n_experts, capacity_factor)
        self.experts = nn.ModuleList([LoRAExpertFFN(d_model, d_hidden, r=r, alpha=alpha) for _ in range(n_experts)])

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, MoEStats]:
        B, T, D = x.shape
        tokens = x.reshape(B*T, D)
        dispatch, combine, stats = self.router(tokens)

        y_tokens = torch.zeros_like(tokens)
        for e, expert in enumerate(self.experts):
            idx = torch.nonzero(dispatch[:, e] > 0.0, as_tuple=False).squeeze(-1)
            if idx.numel() == 0:
                continue
            out_e = expert(tokens[idx])
            w = combine[idx, e].unsqueeze(-1)
            y_tokens[idx] += out_e * w

        return y_tokens.reshape(B, T, D), stats

class TinyMoERegressorLoRA(nn.Module):
    def __init__(self, d_model=48, d_hidden=96, n_experts=4, r=4):
        super().__init__()
        self.n_experts = n_experts
        self.embed = nn.Linear(2 + n_experts, d_model)
        self.moe = LoRA_MoE_Layer(d_model, d_hidden, n_experts=n_experts, r=r, alpha=2*r, capacity_factor=1.25)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x2: torch.Tensor, task: torch.Tensor):
        task_oh = F.one_hot(task, num_classes=self.n_experts).float()
        inp = torch.cat([x2, task_oh], dim=-1).unsqueeze(1)
        h = self.embed(inp)
        h, stats = self.moe(h)
        yhat = self.head(h).squeeze(1)
        return yhat, stats

def count_params(m: nn.Module):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

set_seed(7)
model_lora = TinyMoERegressorLoRA(r=4).to(device)
print("Trainable params (LoRA-MoE demo):", count_params(model_lora))

opt_lora = torch.optim.AdamW(model_lora.parameters(), lr=2e-3)

def train_lora(steps=400, batch_size=256, aux_weight=0.01):
    model_lora.train()
    for step in range(steps):
        x, task, y = sample_batch(batch_size, n_tasks=4)
        yhat, stats = model_lora(x, task)
        mse = F.mse_loss(yhat, y)
        loss = mse + aux_weight * stats.aux_loss
        opt_lora.zero_grad()
        loss.backward()
        opt_lora.step()
        if (step+1) % 100 == 0:
            print(f"step {step+1:4d} | mse {mse.item():.4f} | aux {stats.aux_loss.item():.4f} | load {stats.expert_load.tolist()}")

train_lora()

model_lora.eval()
with torch.no_grad():
    x, task, y = sample_batch(2000, n_tasks=4)
    yhat, stats = model_lora(x, task)
    mse = F.mse_loss(yhat, y)
    print("eval mse:", float(mse), "| load:", stats.expert_load.tolist())

## 4) Tool-Augmented MoE Router for Algebraic Geometry Tasks (MathScy-aligned)

The proposal’s core idea is **MoE LLMs + on-demand tool use** (e.g., computer algebra systems and theorem provers).  
Here we demonstrate a **tool-expert MoE** where:
- **experts are tools** (SymPy capabilities stand in for CAS; Lean/ITP is stubbed)
- a lightweight **router** learns to pick the right tool based on the request text

You can replace the router with:
- a small Transformer classifier
- or an LLM-based router fine-tuned with routing supervision

In [ ]:
# Tool experts (pure python functions)

def tool_simplify(expr: str) -> str:
    x, y, z = sp.symbols('x y z')
    e = sp.sympify(expr)
    return str(sp.simplify(e))

def tool_groebner(polys: List[str], gens: List[str]) -> str:
    gens_syms = sp.symbols(' '.join(gens))
    poly_exprs = [sp.sympify(p) for p in polys]
    G = sp.groebner(poly_exprs, *gens_syms, order='lex')
    return str(G)

def tool_solve_system(polys: List[str], gens: List[str]) -> str:
    gens_syms = sp.symbols(' '.join(gens))
    poly_exprs = [sp.sympify(p) for p in polys]
    sol = sp.solve(poly_exprs, gens_syms, dict=True)
    return str(sol)

def tool_counterexample(statement: str, trials: int = 200) -> str:
    '''
    Very small counterexample search by random sampling (illustrative).
    Accepts an equality like: "x**2 + y**2 = (x+y)**2".
    '''
    x, y = sp.symbols('x y')
    if "=" not in statement:
        return "Unsupported format. Provide 'LHS = RHS'."
    lhs_s, rhs_s = [s.strip() for s in statement.split("=", 1)]
    lhs = sp.sympify(lhs_s)
    rhs = sp.sympify(rhs_s)
    for _ in range(trials):
        xv = random.randint(-5, 5)
        yv = random.randint(-5, 5)
        if lhs.subs({x: xv, y: yv}) != rhs.subs({x: xv, y: yv}):
            return f"Counterexample found: x={xv}, y={yv} gives {lhs.subs({x:xv,y:yv})} != {rhs.subs({x:xv,y:yv})}"
    return "No counterexample found in sampled range."

TOOLS = {
    "simplify": tool_simplify,
    "groebner": tool_groebner,
    "solve_system": tool_solve_system,
    "counterexample": tool_counterexample,
}

# A tiny text router (bag-of-words) in PyTorch
class BoWRouter(nn.Module):
    def __init__(self, vocab: Dict[str, int], n_experts: int):
        super().__init__()
        self.vocab = vocab
        self.linear = nn.Linear(len(vocab), n_experts)

    def featurize(self, text: str) -> torch.Tensor:
        vec = torch.zeros(len(self.vocab))
        for tok in re.findall(r"[a-zA-Z_]+", text.lower()):
            if tok in self.vocab:
                vec[self.vocab[tok]] += 1.0
        return vec

    def forward(self, feats: torch.Tensor) -> torch.Tensor:
        return self.linear(feats)

ROUTE_LABELS = ["simplify", "groebner", "solve_system", "counterexample"]
label_to_id = {k:i for i,k in enumerate(ROUTE_LABELS)}

train_examples = [
    ("simplify x**2 + 2*x + 1", "simplify"),
    ("simplify (x+y)**2 - x**2 - y**2", "simplify"),
    ("compute groebner basis for [x*y - 1, x - y] variables x y", "groebner"),
    ("grobner basis for [x**2 + y**2 - 1, x - y] variables x y", "groebner"),
    ("solve system [x+y-1, x-y-3] variables x y", "solve_system"),
    ("solve polynomial system [x**2 + y - 1, y - x] variables x y", "solve_system"),
    ("find counterexample to x**2 + y**2 = (x+y)**2", "counterexample"),
    ("counterexample for x*y = x + y", "counterexample"),
]

# Build vocab
vocab = {}
for text, _ in train_examples:
    for tok in re.findall(r"[a-zA-Z_]+", text.lower()):
        vocab.setdefault(tok, len(vocab))

router = BoWRouter(vocab=vocab, n_experts=len(ROUTE_LABELS)).to(device)
opt_router = torch.optim.AdamW(router.parameters(), lr=1e-2)

set_seed(7)
for epoch in range(200):
    total = 0.0
    for text, lab in train_examples:
        feats = router.featurize(text).to(device)
        logits = router(feats)
        y = torch.tensor(label_to_id[lab]).to(device)
        loss = F.cross_entropy(logits.unsqueeze(0), y.unsqueeze(0))
        opt_router.zero_grad()
        loss.backward()
        opt_router.step()
        total += float(loss)
    if (epoch+1) % 50 == 0:
        print(f"epoch {epoch+1} | loss {total/len(train_examples):.4f}")

def route(text: str) -> str:
    with torch.no_grad():
        feats = router.featurize(text).to(device)
        logits = router(feats)
        pred = int(torch.argmax(logits).cpu())
    return ROUTE_LABELS[pred]

def run_tool_request(text: str):
    expert = route(text)
    print("Routed to expert:", expert)
    if expert == "simplify":
        expr = text.split("simplify", 1)[-1].strip()
        print(TOOLS[expert](expr))
    elif expert == "groebner":
        polys = re.findall(r"\[(.*?)\]", text)
        polys = polys[0].split(",") if polys else ["x*y-1", "x-y"]
        vars_ = re.findall(r"variables? ([a-z ]+)", text)
        gens = vars_[0].split() if vars_ else ["x", "y"]
        print(TOOLS[expert]([p.strip() for p in polys], gens))
    elif expert == "solve_system":
        polys = re.findall(r"\[(.*?)\]", text)
        polys = polys[0].split(",") if polys else ["x+y-1", "x-y-3"]
        vars_ = re.findall(r"variables? ([a-z ]+)", text)
        gens = vars_[0].split() if vars_ else ["x", "y"]
        print(TOOLS[expert]([p.strip() for p in polys], gens))
    elif expert == "counterexample":
        if "counterexample" in text.lower():
            tail = re.split("counterexample( for| to)?", text, flags=re.IGNORECASE)
            stmt = tail[-1].strip()
        else:
            stmt = text
        stmt = stmt.replace("find", "").replace("to", "").strip()
        print(TOOLS[expert](stmt))
    else:
        print("No tool available.")

run_tool_request("simplify (x+y)**2 - x**2 - y**2")
run_tool_request("compute groebner basis for [x*y - 1, x - y] variables x y")
run_tool_request("solve system [x+y-1, x-y-3] variables x y")
run_tool_request("find counterexample to x**2 + y**2 = (x+y)**2")

### 4.1 (Optional) Hook to Lean 4 / ITP tooling

MathScy targets interactive theorem provers (ITPs). In practice, an MoE expert could be:
- a proof-search LLM expert
- a tool-calling agent that invokes Lean tactics
- a verifier expert that checks proof states

Below is a *minimal* subprocess hook that you can adapt if `lean` is installed in your environment.

In [ ]:
import shutil, subprocess, os, tempfile
from typing import Tuple

def lean_available() -> bool:
    return shutil.which("lean") is not None

def run_lean_file(lean_code: str) -> Tuple[int, str]:
    '''
    Writes a temporary .lean file and runs `lean` on it.
    Returns (exit_code, stdout+stderr).
    '''
    if not lean_available():
        return (127, "Lean not found on PATH. Install Lean 4 to enable this demo.")
    with tempfile.TemporaryDirectory() as td:
        path = os.path.join(td, "Demo.lean")
        with open(path, "w", encoding="utf-8") as f:
            f.write(lean_code)
        p = subprocess.run(["lean", path], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        return (p.returncode, p.stdout)

demo_lean = r'''
-- Minimal Lean 4 check (requires Lean 4 installed)
theorem add_comm (a b : Nat) : a + b = b + a := by
  exact Nat.add_comm a b
'''

code, out = run_lean_file(demo_lean)
print("exit:", code)
print(out[:800])

## 5) Orchestration scaffold (MathScy-style)

MathScy uses:
- multiple specialist models/agents (conjecture / proof / counterexample)
- debate / consensus / verification
- explanations tailored to mathematicians

Below is a minimal orchestration skeleton that you can connect to real MoE LLMs and ITP tools.

In [ ]:
from typing import Callable

class Agent:
    def __init__(self, name: str, fn: Callable[[str], Dict[str, str]]):
        self.name = name
        self.fn = fn

    def run(self, query: str) -> Dict[str, str]:
        out = self.fn(query)
        out["agent"] = self.name
        return out

def conjecture_agent(q: str) -> Dict[str, str]:
    return {"proposal": f"Conjecture sketch for: {q}", "evidence": "Heuristic pattern match (toy)."}

def proof_agent(q: str) -> Dict[str, str]:
    return {"proposal": f"Proof outline for: {q}", "evidence": "Suggest invoking Lean tactics / lemma retrieval (toy)."}

def counterexample_agent(q: str) -> Dict[str, str]:
    if "=" in q:
        return {"proposal": tool_counterexample(q), "evidence": "Random sampling over small integers (toy)."}
    return {"proposal": "Need a concrete equality/claim to test for counterexamples.", "evidence": "Insufficient structure."}

agents = [Agent("conjecture", conjecture_agent),
          Agent("proof", proof_agent),
          Agent("counterexample", counterexample_agent)]

def orchestrate(query: str) -> Dict[str, str]:
    ql = query.lower()
    if "counterexample" in ql or "=" in ql:
        chosen = ["counterexample", "proof"]
    elif "prove" in ql or "theorem" in ql:
        chosen = ["proof", "conjecture"]
    else:
        chosen = ["conjecture", "proof"]

    results = [a.run(query) for a in agents if a.name in chosen]

    final = None
    for r in results:
        if r["agent"] == "counterexample" and "Counterexample found" in r["proposal"]:
            final = r
            break
    if final is None:
        final = next((r for r in results if r["agent"] == "proof"), results[0])

    explanation = (
        f"Routing decision: {chosen}. Selected '{final['agent']}' based on a simple policy "
        f"(prefer concrete counterexamples; otherwise prefer proof guidance)."
    )

    return {
        "query": query,
        "agents_ran": ", ".join([r['agent'] for r in results]),
        "final_agent": final["agent"],
        "final_output": final["proposal"],
        "explanation": explanation,
    }

print(orchestrate("Find a counterexample: x**2 + y**2 = (x+y)**2"))
print()
print(orchestrate("Prove: for all a b : Nat, a + b = b + a"))

## Notes for scaling to the full MathScy system

- Replace toy routers with **LLM routers** trained on routing supervision (which expert/tool solved it best).
- Replace toy experts with **MoE LLMs** (Switch/Top-1, Expert-Choice, or MoE adapters).
- Use **formal feedback** (Lean proof checking) as an outer loop:
  - reward/penalty for tool-calls and proof attempts
  - distill verified proofs into training data for expert specialization
- Add **retrieval** (RAG) for lemma/premise selection and domain corpora.
- Log and surface reasoning artifacts for the **Explainable AI layer**.